In [1]:
import os
import math
import shutil 
import random
import itertools
import numpy as np
import pandas as pd 
from glob import glob

In [2]:
train_csv_file="fold1_train.csv" 
test_csv_file="fold1_test.csv" 

path="Datasets/1.HAM10000/"
data_root="rasm/siamese/fold1/"

In [3]:
output_filename_train_genuine = "ham10000_siamese_train_genuine_pairs.csv"
output_filename_train_imposter = "ham10000_siamese_train_imposter_pairs.csv"  
output_filename_test_genuine = "ham10000_siamese_test_genuine_pairs.csv"
output_filename_test_imposter = "ham10000_siamese_test_imposter_pairs.csv"  

In [4]:
transform_list = ['AutoContrast', 'BoxBlur', 'CLAHE', 'ChannelShuffle', 'ChromaticAberration', 'CoarseDropout', 
                  'ColorJitter', 'Defocus', 'Dilation', 'ElasticTransform', 'Emboss', 'Equalize', 'EqualizeYUV', 
                  'Erosion', 'Frost', 'Gamma', 'GaussNoise', 'GaussianBlur', 'GlassBlur', 'HorizontalFlip', 
                  'HueSaturationValue', 'Invert', 'JPEGCompression', 'MedianBlur', 'MotionBlur', 'OpticalDistortion', 
                  'Original', 'Perspective', 'Posterize', 'RGBShift', 'RandomBrightnessContrast', 'RandomFog', 
                  'RandomGravel', 'RandomGridShuffle', 'RandomRain', 'RandomShadow', 'RandomSnow', 'RandomSunFlare', 
                  'RandomToneCurve', 'Rotate', 'SaltandPepper', 'Scale', 'Sharpen', 'ShearX', 'ShearY', 'ShiftScaleRotate', 
                  'Solarize', 'Speckle', 'ToGray', 'ToSepia', 'Translate', 'VerticalFlip', 'Vignette', 'ZoomBlur']

print(len(transform_list))

54


In [5]:
df_train=pd.read_csv(data_root+train_csv_file, header=None) 
train_file_list=df_train[0].tolist()

In [ ]:
# Genuine pairs
train_folderlist_np=np.array(glob("rasm/siamese/fold1/train_aug/*/*"))
np.random.shuffle(train_folderlist_np)

train_genuine_pairs=[]
for x in train_folderlist_np:
    file_np=np.array(glob(x+"/*"))
    np.random.shuffle(file_np)
    comb=list(itertools.combinations_with_replacement(file_np, 2))
    random.shuffle(comb)
    train_genuine_pairs.append(comb)
    
    x1=x.split('.')[0]
    
    # Generate filenames
    filenames = []
    for i in transform_list:
        filenames.append(f"{x1}_{i}.jpg")
    
    # Generate all unique pairs (combinations of 2) and store in a list
    image_combinations = list(itertools.combinations(filenames, 2))  
    
    # Perform uniform random sampling 
    sample_size = int(len(image_combinations)*0.25)  
    random_sample = random.sample(image_combinations, sample_size)
    
    train_genuine_pairs.append(random_sample)

train_genuine_pairs = np.array(train_genuine_pairs)
train_genuine_pairs = train_genuine_pairs.reshape(train_genuine_pairs.shape[0]*train_genuine_pairs.shape[1],train_genuine_pairs.shape[2])
np.random.shuffle(train_genuine_pairs)

print('train genuine pairs shape : ', train_genuine_pairs.shape)

In [ ]:
same=0
for i in range(train_genuine_pairs.shape[0]):
    if train_genuine_pairs[i][0].split('_')[1] != train_genuine_pairs[i][1].split('_')[1]:
        same = same + 1

print(same)

In [ ]:
num_selected_images=len(random_sample)
print("num selected images : ", num_selected_images)

num_total_images=len(image_combinations)
print("num total images : ", num_total_images)

In [ ]:
# Imposter pairs
train_imposter_pairs=[]
for x in train_file_list:
    c=x.split('/')[3]    
    x1=x.split('.')[0]
    
    # Generate filenames
    filenames = []
    for i in transform_list:
        filenames.append(f"{x1}_{i}.jpg")

    current_folder_filenames=filenames
    current_folder_filenames_repeated=np.tile(current_folder_filenames,50)  
    np.random.shuffle(current_folder_filenames_repeated)
    current_folder_filenames_repeated=current_folder_filenames_repeated[:num_selected_images]

    tfl = np.array(train_file_list)
    other_folderlist=[]
    for t in tfl:
        cl=t.split('/')[3]     
        if c != cl:
            other_folderlist.append(t)
    
    temp = []
    for z in other_folderlist:
        x2=z.split('.')[0]
        for i in transform_list:
            temp.append(f"{x2}_{i}.jpg")

    temp=np.array(temp)
    temp=temp.reshape(-1)
    np.random.shuffle(temp)
    temp=temp[:num_selected_images]
    other_folder_filenames=temp

    train_imposter_pairs.append(np.column_stack((current_folder_filenames_repeated, other_folder_filenames)))

train_imposter_pairs=np.array(train_imposter_pairs)
train_imposter_pairs = train_imposter_pairs.reshape(train_imposter_pairs.shape[0]*train_imposter_pairs.shape[1],train_imposter_pairs.shape[2])
np.random.shuffle(train_imposter_pairs)  

print('train imposter pairs shape : ', train_imposter_pairs.shape)

In [ ]:
# Train Labels and Save
labels_train_genuine=np.zeros(train_genuine_pairs.shape[0], dtype=np.int8)
labels_train_genuine=labels_train_genuine.reshape(-1,1)
train_genuine_pairs = np.concatenate((train_genuine_pairs, labels_train_genuine), axis=1)

labels_train_imposter=np.ones(train_imposter_pairs.shape[0], dtype=np.int8)
labels_train_imposter=labels_train_imposter.reshape(-1,1)
train_imposter_pairs = np.concatenate((train_imposter_pairs, labels_train_imposter), axis=1)

In [ ]:
# check same file possibility
same=0
for i in range(train_imposter_pairs.shape[0]):
    if train_imposter_pairs[i][0].split("_")[1] == train_imposter_pairs[i][1].split("_")[1]:
        same=same+1

print('same file duplicated in imposter pairs: ', same)

# check same class possibility
same=0
for i in range(train_imposter_pairs.shape[0]):
    if train_imposter_pairs[i][0].split("/")[3] == train_imposter_pairs[i][1].split("/")[3]:
        same=same+1

print('same class duplicated in imposter pairs : ', same)

In [ ]:
# save csv files
df=pd.DataFrame(train_genuine_pairs)
df.to_csv(data_root+output_filename_train_genuine, index=False, header=False)   
df=pd.DataFrame(train_imposter_pairs)
df.to_csv(data_root+output_filename_train_imposter, index=False, header=False)  

In [ ]:
df_test=pd.read_csv(data_root+test_csv_file, header=None)
test_file_list=df_test[0].tolist()

In [ ]:
# Genuine pairs
test_genuine_pairs=[]
for x in test_file_list:
    x1=x.split('.')[0]
    
    # Generate filenames
    filenames = []
    for i in transform_list:
        filenames.append(f"{x1}_{i}.jpg")
    
    # Generate all unique pairs (combinations of 2) and store in a list
    image_combinations = list(itertools.combinations(filenames, 2))  
    
    # Perform uniform random sampling 
    sample_size = int(len(image_combinations)*0.4)  
    random_sample = random.sample(image_combinations, sample_size)
    
    test_genuine_pairs.append(random_sample)

test_genuine_pairs = np.array(test_genuine_pairs)
test_genuine_pairs = test_genuine_pairs.reshape(test_genuine_pairs.shape[0]*test_genuine_pairs.shape[1],test_genuine_pairs.shape[2])
np.random.shuffle(test_genuine_pairs)

print('test genuine pairs shape : ', test_genuine_pairs.shape)

In [ ]:
same=0
for i in range(test_genuine_pairs.shape[0]):
    if test_genuine_pairs[i][0].split('_')[1] != test_genuine_pairs[i][1].split('_')[1]:
        same = same + 1

print(same)

In [ ]:
num_selected_images=len(random_sample)
print("num selected images : ", num_selected_images)

num_total_images=len(image_combinations)
print("num total images : ", num_total_images)

In [ ]:
# Imposter pairs
test_imposter_pairs=[]
for x in test_file_list:
    c=x.split('/')[3]    
    x1=x.split('.')[0]
    
    # Generate filenames
    filenames = []
    for i in transform_list:
        filenames.append(f"{x1}_{i}.jpg")

    current_folder_filenames=filenames
    current_folder_filenames_repeated=np.tile(current_folder_filenames,50)  
    np.random.shuffle(current_folder_filenames_repeated)
    current_folder_filenames_repeated=current_folder_filenames_repeated[:num_selected_images]

    tfl = np.array(test_file_list)
    other_folderlist=[]
    for t in tfl:
        cl=t.split('/')[3]     
        if c != cl:
            other_folderlist.append(t)
    
    temp = []
    for z in other_folderlist:
        x2=z.split('.')[0]
        for i in transform_list:
            temp.append(f"{x2}_{i}.jpg")

    temp=np.array(temp)
    temp=temp.reshape(-1)
    np.random.shuffle(temp)
    temp=temp[:num_selected_images]
    other_folder_filenames=temp

    test_imposter_pairs.append(np.column_stack((current_folder_filenames_repeated, other_folder_filenames)))

test_imposter_pairs=np.array(test_imposter_pairs)
test_imposter_pairs=test_imposter_pairs.reshape(test_imposter_pairs.shape[0]*test_imposter_pairs.shape[1], test_imposter_pairs.shape[2])
np.random.shuffle(test_imposter_pairs)  

print('test imposter pairs shape : ', test_imposter_pairs.shape)

In [ ]:
# Test Labels and Save
labels_test_genuine=np.zeros(test_genuine_pairs.shape[0], dtype=np.int8)
labels_test_genuine=labels_test_genuine.reshape(-1,1)
test_genuine_pairs=np.concatenate((test_genuine_pairs, labels_test_genuine), axis=1)

labels_test_imposter=np.ones(test_imposter_pairs.shape[0], dtype=np.int8)
labels_test_imposter=labels_test_imposter.reshape(-1,1)
test_imposter_pairs = np.concatenate((test_imposter_pairs, labels_test_imposter), axis=1)

In [ ]:
# check same file possibility
same=0
for i in range(test_imposter_pairs.shape[0]):
    if test_imposter_pairs[i][0].split("_")[1] == test_imposter_pairs[i][1].split("_")[1]:
        same=same+1

print('same file duplicated in imposter pairs: ', same)

# check same class possibility
same=0
for i in range(test_imposter_pairs.shape[0]):
    if test_imposter_pairs[i][0].split("/")[3] == test_imposter_pairs[i][1].split("/")[3]:
        same=same+1

print('same class duplicated in imposter pairs : ', same)

In [ ]:
# save csv files
df=pd.DataFrame(test_genuine_pairs)
df.to_csv(data_root+output_filename_test_genuine, index=False, header=False)  
df=pd.DataFrame(test_imposter_pairs)
df.to_csv(data_root+output_filename_test_imposter, index=False, header=False)